In [9]:
import pandas as pd
import re
import nltk

# Download required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')   
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

print(" Libraries imported!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\reach\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\reach\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\reach\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\reach\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


 Libraries imported!


In [10]:
df_train = pd.read_csv("../data/resume_jd_train.csv")
df_test = pd.read_csv("../data/resume_jd_test.csv")

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print("\nFirst row label:", df_train['label'][0])

Train shape: (6241, 3)
Test shape: (1759, 3)

First row label: No Fit


In [11]:
print("=== MISSING VALUES ===")
print(df_train.isnull().sum())

print("\n=== DUPLICATE ROWS ===")
print(f"Duplicates: {df_train.duplicated().sum()}")

=== MISSING VALUES ===
resume_text             0
job_description_text    0
label                   0
dtype: int64

=== DUPLICATE ROWS ===
Duplicates: 1


In [12]:
# Remove the duplicate
df_train = df_train.drop_duplicates()

print(" Duplicate removed!")
print("New train shape:", df_train.shape)

 Duplicate removed!
New train shape: (6240, 3)


In [13]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove emails
    text = re.sub(r'\S+@\S+', '', text)
    
    # 3. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # 4. Remove phone numbers
    text = re.sub(r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b', '', text)
    
    # 5. Remove special characters
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    
    # 6. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 7. Tokenize
    tokens = word_tokenize(text)
    
    # 8. Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(word) 
              for word in tokens 
              if word not in stop_words 
              and len(word) > 2]
    
    return ' '.join(tokens)

print(" Cleaning function defined!")

 Cleaning function defined!


In [14]:
def remove_gender_signals(text):
    # Remove gendered pronouns and titles
    patterns = r'\b(he|she|him|her|his|hers|mr|ms|mrs|miss|male|female)\b'
    text = re.sub(patterns, '[MASKED]', text, flags=re.IGNORECASE)
    return text

print(" Gender masking function defined!")

 Gender masking function defined!


In [15]:
print("Cleaning resumes... please wait ⏳")
df_train['resume_clean'] = df_train['resume_text'].apply(remove_gender_signals).apply(clean_text)

print("Cleaning JDs... please wait ⏳")
df_train['jd_clean'] = df_train['job_description_text'].apply(clean_text)

# Same for test set
print("Cleaning test set... please wait ⏳")
df_test['resume_clean'] = df_test['resume_text'].apply(remove_gender_signals).apply(clean_text)
df_test['jd_clean'] = df_test['job_description_text'].apply(clean_text)

print("✅ Cleaning complete!")

Cleaning resumes... please wait ⏳
Cleaning JDs... please wait ⏳
Cleaning test set... please wait ⏳
✅ Cleaning complete!


In [16]:
print("=== ORIGINAL RESUME (first 300 chars) ===")
print(df_train['resume_text'][0][:300])

print("\n=== CLEANED RESUME (first 300 chars) ===")
print(df_train['resume_clean'][0][:300])

print("\n=== ORIGINAL JD (first 300 chars) ===")
print(df_train['job_description_text'][0][:300])

print("\n=== CLEANED JD (first 300 chars) ===")
print(df_train['jd_clean'][0][:300])

=== ORIGINAL RESUME (first 300 chars) ===
SummaryHighly motivated Sales Associate with extensive customer service and sales experience. Outgoing sales professional with track record of driving increased sales, improving buying experience and elevating company profile with target market.
Highlights-Soft Skills: Public Speaking, Public Relati

=== CLEANED RESUME (first 300 chars) ===
summaryhighly motivated sale associate extensive customer service sale experience outgoing sale professional track record driving increased sale improving buying experience elevating company profile target market highlight soft skill public speaking public relation team building project management p

=== ORIGINAL JD (first 300 chars) ===
Net2Source Inc. is an award-winning total workforce solutions company recognized by Staffing Industry Analysts for our accelerated growth of 300% in the last 3 years with over 5500+ employees globally, with over 30+ locations in the US and global operations in 32 countries. 

In [18]:
# Show MORE characters to find where pronoun is
print("ORIGINAL (full text snippet with pronoun):")

# Search for the actual pronoun location
text = df_train['resume_text'][idx]
# Find where the pronoun is
import re
matches = [(m.start(), m.group()) for m in re.finditer(r'\b(he|she|him|her|his)\b', text, re.IGNORECASE)]
print("Pronouns found at positions:", matches)

# Show context around first match
if matches:
    pos = matches[0][0]
    print("\nContext around pronoun:")
    print(text[max(0,pos-100):pos+100])

ORIGINAL (full text snippet with pronoun):
Pronouns found at positions: [(4289, 'his')]

Context around pronoun:
otype ahead of schedule and presented
          the project to the Manufacturing Vice President and his staff in the company
          headquarters.Responsibilities included:
          Met with suppli


In [19]:
df_train.to_csv("../data/resume_jd_train_cleaned.csv", index=False)
df_test.to_csv("../data/resume_jd_test_cleaned.csv", index=False)

print("✅ Cleaned dataset saved!")
print("\nNew columns added:")
print(df_train.columns.tolist())

✅ Cleaned dataset saved!

New columns added:
['resume_text', 'job_description_text', 'label', 'resume_clean', 'jd_clean']
